# PiEEG Analysis Notebook


**Created:** 2026-06-12T13:05:36.637247
**Stream:** Left_PiEEG
**Channels:** 8 (Ch0, Ch1, Ch2, Ch3, ...)
**Sampling Rate:** 250 Hz
**Mains Frequency:** 50.0 Hz

---


# Brain Data Analysis

This notebook visualizes your current EEG data including band powers, neural states, and signal quality.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datetime import datetime

# Set up plotting style
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

In [ ]:
# Get current neural state data
from pieeg_copilot import get_neural_state, get_band_powers, get_channel_quality

# Fetch current brain data
neural_state = get_neural_state()
band_powers = get_band_powers(per_channel=True)
channel_quality = get_channel_quality()

print("Current Neural State:")
print(f"Focus: {neural_state['focus']:.3f}")
print(f"Relax: {neural_state['relax']:.3f}")
print(f"Engagement: {neural_state['engagement']:.3f}")
print(f"Dominant Band: {neural_state['dominant_band']}")
print(f"Signal Quality: {neural_state['signal_quality']:.3f}")
print(f"Warming Up: {neural_state['warming_up']}")

In [ ]:
# Create comprehensive brain data visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Real-Time Brain Data Analysis', fontsize=16, fontweight='bold')

# 1. Band Powers Bar Chart
ax1 = axes[0, 0]
bands = ['Delta', 'Theta', 'Alpha', 'Beta', 'Gamma']
band_values = [band_powers['delta'], band_powers['theta'], band_powers['alpha'], 
               band_powers['beta'], band_powers['gamma']]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']

bars = ax1.bar(bands, band_values, color=colors, alpha=0.8)
ax1.set_title('EEG Band Powers', fontweight='bold')
ax1.set_ylabel('Relative Power')
ax1.set_ylim(0, max(band_values) * 1.1)

# Add value labels on bars
for bar, value in zip(bars, band_values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{value:.3f}', ha='center', va='bottom', fontweight='bold')

# 2. Neural State Indicators
ax2 = axes[0, 1]
states = ['Focus', 'Relax', 'Engagement']
state_values = [neural_state['focus'], neural_state['relax'], neural_state['engagement']]
state_colors = ['#E74C3C', '#2ECC71', '#3498DB']

bars2 = ax2.bar(states, state_values, color=state_colors, alpha=0.8)
ax2.set_title('Neural State Indices', fontweight='bold')
ax2.set_ylabel('Level (0-1)')
ax2.set_ylim(0, 1)

# Add value labels
for bar, value in zip(bars2, state_values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{value:.3f}', ha='center', va='bottom', fontweight='bold')

# 3. Per-Channel Band Powers Heatmap
ax3 = axes[1, 0]
if 'per_channel' in band_powers:
    channel_data = band_powers['per_channel']
    channels = list(channel_data.keys())
    bands = ['delta', 'theta', 'alpha', 'beta', 'gamma']
    
    # Create matrix for heatmap
    heatmap_data = []
    for channel in channels:
        row = [channel_data[channel][band] for band in bands]
        heatmap_data.append(row)
    
    im = ax3.imshow(heatmap_data, cmap='viridis', aspect='auto')
    ax3.set_xticks(range(len(bands)))
    ax3.set_xticklabels([b.capitalize() for b in bands])
    ax3.set_yticks(range(len(channels)))
    ax3.set_yticklabels(channels)
    ax3.set_title('Per-Channel Band Powers', fontweight='bold')
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax3)
    cbar.set_label('Relative Power')
else:
    ax3.text(0.5, 0.5, 'Per-channel data\nnot available', 
             ha='center', va='center', transform=ax3.transAxes)
    ax3.set_title('Per-Channel Band Powers', fontweight='bold')

# 4. Signal Quality Overview
ax4 = axes[1, 1]
if channel_quality:
    channels = list(channel_quality.keys())
    quality_scores = [channel_quality[ch]['score'] for ch in channels]
    quality_colors = ['#E74C3C' if score < 0.5 else '#F39C12' if score < 0.8 else '#2ECC71' 
                     for score in quality_scores]
    
    bars4 = ax4.bar(channels, quality_scores, color=quality_colors, alpha=0.8)
    ax4.set_title('Channel Signal Quality', fontweight='bold')
    ax4.set_ylabel('Quality Score (0-1)')
    ax4.set_ylim(0, 1)
    ax4.tick_params(axis='x', rotation=45)
    
    # Add horizontal lines for quality thresholds
    ax4.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Poor')
    ax4.axhline(y=0.8, color='orange', linestyle='--', alpha=0.5, label='Good')
    ax4.legend()
else:
    ax4.text(0.5, 0.5, 'Channel quality\ndata not available', 
             ha='center', va='center', transform=ax4.transAxes)
    ax4.set_title('Channel Signal Quality', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Additional analysis: Spectral details
from pieeg_copilot import analyze_spectrum

spectrum_data = analyze_spectrum()
print("\nSpectral Analysis:")
print(f"Individual Alpha Frequency: {spectrum_data.get('alpha_peak_freq', 'N/A')} Hz")
print(f"Aperiodic Slope: {spectrum_data.get('aperiodic_slope', 'N/A')}")
print(f"Theta/Beta Ratio: {spectrum_data.get('theta_beta_ratio', 'N/A')}")
print(f"Spectral Entropy: {spectrum_data.get('spectral_entropy', 'N/A')}")
if 'frontal_alpha_asymmetry' in spectrum_data:
    print(f"Frontal Alpha Asymmetry: {spectrum_data['frontal_alpha_asymmetry']}")

In [ ]:
# Summary statistics
print("\n" + "="*50)
print("BRAIN DATA SUMMARY")
print("="*50)
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\nDominant frequency band: {neural_state['dominant_band']}")
print(f"Overall signal quality: {neural_state['signal_quality']:.3f}")
print(f"System status: {'Warming up' if neural_state['warming_up'] else 'Ready'}")

# Find strongest band
max_band = max(band_values)
max_band_name = bands[band_values.index(max_band)]
print(f"Strongest band: {max_band_name} ({max_band:.3f})")

# Neural state summary
max_state = max(state_values)
max_state_name = states[state_values.index(max_state)]
print(f"Dominant neural state: {max_state_name} ({max_state:.3f})")

print("\nNote: All values are relative to your current session range.")
if neural_state['warming_up']:
    print("⚠️  System is still warming up - readings may be settling.")
if neural_state['signal_quality'] < 0.7:
    print("⚠️  Signal quality is suboptimal - check electrode connections.")